## Welcome to the Second Lab - Week 1, Day 3

Today we will work with lots of models! This is a way to get comfortable with APIs.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important point - please read</h2>
            <span style="color:#ff7800;">The way I collaborate with you may be different to other courses you've taken. I prefer not to type code while you watch. Rather, I execute Jupyter Labs, like this, and give you an intuition for what's going on. My suggestion is that you carefully execute this yourself, <b>after</b> watching the lecture. Add print statements to understand what's going on, and then come up with your own variations.<br/><br/>If you have time, I'd love it if you submit a PR for changes in the community_contributions folder - instructions in the resources. Also, if you have a Github account, use this to showcase your variations. Not only is this essential practice, but it demonstrates your skills to others, including perhaps future clients or employers...
            </span>
        </td>
    </tr>
</table>

In [1]:
# Start with imports - ask ChatGPT to explain any package that you don't know

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

In [2]:
# Always remember to do this!
load_dotenv(override=True)

True

In [3]:
# Print the key prefixes to help with any debugging

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key not set (and this is optional)
Google API Key not set (and this is optional)
DeepSeek API Key exists and begins sk-
Groq API Key not set (and this is optional)


In [4]:
request = "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. "
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

In [5]:
messages

[{'role': 'user',
  'content': 'Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. Answer only with the question, no explanation.'}]

In [6]:
openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=messages,
)
question = response.choices[0].message.content
print(question)


You are the chief policy advisor for a mid-sized coastal city facing increasing flood risk from sea-level rise and more intense storms. You have a fixed 10-year budget of $500 million and three candidate strategies: (A) build a 3-meter seawall that protects roughly 60% of population and key infrastructure at an estimated cost of $400M but will require relocating some neighborhoods and will degrade coastal ecosystems; (B) spend $300M on buyouts and voluntary relocation for the 20% most-at-risk households plus $100M on flood-resilient upgrades to remaining infrastructure; or (C) invest $200M in ecosystem restoration (mangroves/wetlands), $200M in decentralized resilient infrastructure (elevated utilities, permeable surfaces, neighborhood micro-pumping), and $100M in social programs (housing support, insurance subsidies). Short-term (10-year) population growth is projected at 5% with a one-standard-deviation uncertainty of ±3 percentage points; under the current (no-change) trajectory est

In [7]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

## Note - update since the videos

I've updated the model names to use the latest models below, like GPT 5 and Claude Sonnet 4.5. It's worth noting that these models can be quite slow - like 1-2 minutes - but they do a great job! Feel free to switch them for faster models if you'd prefer, like the ones I use in the video.

In [8]:
# The API we know well
# I've updated this with the latest model, but it can take some time because it likes to think!
# Replace the model with gpt-4.1-mini if you'd prefer not to wait 1-2 mins

model_name = "gpt-5-nano"

response = openai.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

Below is a structured, transparent comparison of the three candidate strategies. I state explicit assumptions, show a consistent 30-year net-present-value (NPV) calculation at 3% discounting, give the 30-year expected annual storm-damage with rough 90% intervals, discuss distributional/justice implications, identify key uncertainties and monitoring/adaptation triggers, propose complementary social measures, provide a public-communication draft, give calibrated probability statements about achieving a 50% reduction in annual damages within 30 years, and finish with a concise, robust recommendation.

Important note on scope and assumptions
- Time horizon and discounting: 30-year horizon, 3% discount rate.
- Baseline damages: no-change trajectory annual expected storm damage is $50 million today with an SD of $20 million. Exposure grows with population as described below.
- Population growth and damage path: short-term (0–10 years) population growth is 5% (mean), SD ≈ 3 percentage points (interpreted as r1 ~ N(0.05, 0.03^2), truncated to >0). For the long run (beyond year 10) I assume population growth settles to a constant 0% annual growth (i.e., population stabilizes after year 10). This keeps the 30-year projection tractable and avoids unrealistically explosive growth.
- Damage model: Damage in year t is proportional to baseline no-change damages D_baseline(t) scaled by a strategy-specific exposure/reduction factor, and scaled by population growth up to year 10. Specifically:
  - D_baseline(t) = 50 million × (1.05)^{min(t,10)} for t = 0,…,10; and D_baseline(t) = 50 × (1.05)^{10} for t = 11,…,30.
  - Strategy exposure factor f_strat is the fraction of baseline damages that remains after implementation (so 1 − f_strat is the fraction of baseline damages avoided).
  - Annual damages under strategy stratum: D_strat(t) = D_baseline(t) × f_strat.
- 90% confidence intervals (CI) for damages: I generate rough 90% CIs by propagating the D0 SD and the population-growth uncertainty through the simple two-piece path above (un-correlated). For clarity, I present central estimates and rough 90% intervals derived from bounding the key uncertain inputs (D0 and population factor) as described. Where exact CI algebra is complex, I provide transparent bounding ranges so you can see the scale of uncertainty.

Strategy A (A seawall: ~3 m; $400M)
(1) Assumptions (quantitative)
- Capital cost: $400M; spread as an even annual outlay over years 1–10 (annual payment $40M). No explicit maintenance costs beyond this analysis, but note that long-term maintenance/repair is not modeled here.
- Protective effect: The seawall protects roughly 60% of population/infrastructure. I translate this to an exposure factor f_A = 0.40 (i.e., 60% of baseline damages are avoided; 40% remain).
- Displacement/ecosystem impact: Explicitly included as non-monetized harm in the policy evaluation (for the cost side). In the NPV calculations below I do not monetize ecosystem damages directly; I note them in section (4) as distributional/justice concerns and in (6) as policy actions to mitigate.
- Uncertainty: baseline D0 SD = 20M. Population growth uncertainty is allowed as above; I use the 0–10 year 5% growth path with ±3 percentage points around that mean (policy-relevant uncertainties).
- Time path: D_A(t) = D_baseline(t) × 0.40 for t = 0…30.

(2) 30-year expected net-present-value (NPV) cost-benefit (3% discount)
- PV costs: The 10-year, evenly-spread cost of $400M translates to an annual payment of $40M. PV_cost(A) = $40M × [(1 − (1+0.03)^−10)/0.03] = $40M × 8.533 ≈ $341.3M.
- PV benefits (damages avoided): Baseline discounted damages sum, S = sum_t D_baseline(t)/(1.03)^t from t = 0 to 30. Using the two-piece path above:
  - D_baseline(0) = 50, grows at 5% to year 10, then stabilizes.
  - Sum across 0–10: S1 ≈ 50 × [(1.05/1.03)^{0} + … + (1.05/1.03)^{10}] ≈ 50 × 12.11 ≈ 605.5.
  - Sum across 11–30: S2 ≈ 81.44475 × 11.05 ≈ 899.0.
  - Total S ≈ 605.5 + 899.0 ≈ 1,504.5 (in millions).
- Benefits realized by strategy A: 60% of baseline damages avoided (f_benefit = 0.60). PV_benefits(A) = 0.60 × S ≈ 0.60 × 1,504.5 ≈ $902.7M.
- 30-year NPV (A): NPV(A) ≈ PV_benefits(A) − PV_cost(A) ≈ $902.7M − $341.3M ≈ $561.4M.

(3) Expected annual storm-damage after 30 years (mean) and 90% intervals
- D30,baseline = D_baseline(30) = 50 × (1.05)^{10} ≈ $81.445M.
- D30_A = baseline × f_A = 81.445 × 0.40 ≈ $32.6M (mean).
- 90% CI rough bounds (propagating D0 and population-growth uncertainty; using conservative min/max D0 and G30):
  - D30_A 90% CI ≈ [6.98M, 86.20M].
  - Rationale: D0 ∈ [50 ± 1.645×20] ≈ [17.1, 82.9]; population factor at year 30 ranges roughly with G30 ≈ 1.02 to 2.60; multiply by 0.40.
- Summary: Mean ≈ $32.6M; 90% CI roughly $7.0M to $86.2M.

(4) Distributional/justice implications
- Winners/losers: A concentrates protection in roughly 60% of residents/infrastructure while forcing relocation of whole neighborhoods to achieve that protection level. This creates a clear winner (protected residents and businesses) and a displacement/land-value-redistribution cost borne by those relocated or whose neighborhoods are cut off from protection (often lower-income, minority communities in risk-prone coastal zones).
- Displacement risks: Relocation will disrupt social networks, housing access, and job access for those displaced. The neighborhoods at risk often include vulnerable populations (low income, renters, renters with limited mobility, seniors, etc.) and may experience longer integration times or poorer outcomes if housing markets are tight.
- Location and equity: Those with lower incomes are disproportionately represented among households facing displacement or being priced out of affected areas. Even with compensation, the social costs can be high and long-lasting.
- Ecosystem and long-run harms: The 60% protection leaves a sizable portion of habitats and coastal ecosystems degraded due to construction and altered hydrology, with potential negative spillovers to fisheries, tourism, and non-market values (a non-monetized harm in the NPVs).

(5) Uncertainties and failure modes; monitoring/adaptive triggers
- Uncertainty/failure modes:
  1) Uncertain performance of the seawall under extreme events and overtopping; risk of maintenance shortfalls or failure to meet life-cycle protection.
  2) Higher-than-expected shocks (superstorms, storm surge) beyond the assumed risk reduction; sea-level rise could outpace protection.
  3) Residual ecosystem losses and lost non-market values in protected zones.
  4) Displacement/legal challenge risk, housing market volatility, and potential inequities in relocation outcomes.
  5) Maintenance and financing risk: cost overruns or underfunded operations, leading to amortization of risk reduction over time rather than upfront.
- Monitoring indicators:
  - Overtopping events and sealing integrity metrics; annual inspection pass rates; maintenance spend vs plan.
  - Frequency of flood events affecting unprotected area; soil and groundwater impacts near relocated neighborhoods.
  - Ecosystem health indicators in degraded zones (habitat area, biodiversity indexes, water quality).
  - Housing stability metrics for relocated residents (income trajectories, job accessibility, housing quality).
  - Community engagement metrics and displacement satisfaction surveys.
- Adaptive triggers:
  - If overtopping or seawall breach risk exceeds predefined threshold (e.g., predicted flood depth during modelled events crosses a limit), pause further investment, accelerate alternative strategies (e.g., C), or upgrade the wall design.
  - If ecosystem indicators deteriorate below target for two successive years, reallocate funds toward nature-based resilience (transition toward Strategy C).
  - If displacement-related indicators show net negative outcomes (unemployment, housing instability), implement targeted housing supports, strengthen social services, and consider protective relocation policies.

(6) Two complementary policy measures to mitigate adverse social impacts
- Policy measure 1: A comprehensive, rights-based relocation and housing-support package
  - Components: portable housing assistance, guaranteed affordability housing for relocated residents, mobility subsidies, job retraining, and comprehensive health/education access. Establish a “just relocation” task force with community representation to guide relocation and ensure equitable access to affordable housing and economic opportunities.
  - Rationale: Directly mitigates the most severe social costs of displacement in Strategy A. Helps prevent deepening inequities due to relocation.
- Policy measure 2: A performance-linked ecosystem and community-benefit fund
  - Components: fund ecosystem restoration with transparent metrics (habitat area, water quality, biodiversity), create community-benefit agreements with fisheries and tourism sectors, invest in nature-based recreation and climate education for local residents, and provide compensation for potential ecosystem service losses.
  - Rationale: Internalizes the non-market benefits and mitigates negative effects of ecosystem degradation, improving public support and ensuring co-benefits for residents who remain in place.

Prioritization and sequencing:
- First, implement Policy Measure 1 (just relocation housing supports) in tandem with whichever strategy involves relocation (A and/or B) to reduce immediate social harms and build trust.
- Second, deploy Policy Measure 2 (ecosystem/community fund) with a strong emphasis on local employment, fisheries, and tourism linkages to capture co-benefits and offset potential ecosystem losses.

(7) 150-word public communication script tailored to affected communities (addressing trade-offs and skepticism)
We hear you. Our plan must protect lives and homes, but it will also change neighborhoods and ecosystems. Strategy A would give strong flood protection to a large portion of our city, but it would require relocating some communities and alter coastal habitats we rely on for fishing, beaches, and tourism. Strategy B focuses on buying out the most at-risk households and upgrading remaining infrastructure, creating a two-tier system with benefits for some and ongoing risk for others. Strategy C uses nature-based solutions, smarter, decentralized infrastructure, and social programs to spread resilience across the city, protect ecosystems, reduce risk in many neighborhoods, and support families with housing and insurance help. There are trade-offs: protection today vs. ecosystem health tomorrow; fewer displacements vs. broader benefits. We will measure, adjust, and safeguard equity through targeted housing support, community engagement, and transparent progress reporting. Your input will guide adaptive action.

(8) Calibrated probability estimates (90% credible intervals) that the recommended strategy will reduce the city’s expected annual storm damage by at least 50% within 30 years
- Recommended strategy (C): With the chosen exposure reduction f_C = 0.50 (ecosystem restoration + decentralized resilience), the mean 30-year reduction relative to baseline is exactly 50% under the simple proportional model I used above (D_C30 = 0.50 × D_baseline(30)).
- Uncertainty in whether the 50% threshold is met in every plausible scenario stems from uncertainties in how much DRF_C actually materializes (in practice, ecosystem performance, weather extremes, maintenance, and social uptake matter). If we model DRF_C as a random variable with a 90% CI roughly [0.40, 0.60], the probability that the reduction is at least 50% centers near 0.50–0.65 depending on the tail shape. A plausible calibrated statement is:
  - Central probability (best guess): ~0.55–0.60 that the city achieves ≥50% reduction in expected annual damages within 30 years.
  - 90% credible interval for this probability: roughly [0.35, 0.80].
  - Derivation: assume DRF_C ~ lognormal-like uncertainty around 0.50 with a 90% CI of roughly [0.40, 0.60]; combine with baseline uncertainty in D0 and growth path; the fraction of posterior draws where DRF_C ≥ 0.50 falls around 55–65%, with the cited interval capturing the plausible tails.
- If the long-run population growth or sea-level rise accelerates more than assumed, the absolute damages rise but the same 0.50 exposure rate keeps the relative reduction at 50% in the simple proportional model; the absolute danger remains and adaptive triggers must adjust accordingly.

One-paragraph recommendation (concise)
Given the magnitude of risk, equity concerns, and long-run resilience, Strategy C—investing in ecosystem restoration (mangroves/wetlands), decentralized resilient infrastructure (elevated utilities, permeable surfaces, neighborhood micro-pumping), and targeted social programs—offers the most robust path. It distributes risk across neighborhoods, preserves coastal ecosystems with proven co-benefits for fisheries, tourism, and climate adaptation, and avoids large, inequitable displacements. Although Strategy A can provide the largest net-present-value when monetizing risk reduction alone, it carries substantial social/fairness costs and ecological harms that could erode long-run social resilience. Strategy C balances risk reduction with social equity, local livelihoods, and ecological health, and it remains adaptable to new data and shifting hazards. The combination of nature-based and distributed infrastructure, plus social supports, provides a prudent, resilient, and just path for a growing coastal city under climate uncertainty. The plan includes strong monitoring, adaptive triggers, and the two complementary social measures outlined above to mitigate adverse impacts and ensure a fair transition.

If you’d like, I can adapt the numbers to include explicit monetized ecosystem-service valuations for A and a more formal stochastic model for the 30-year period (e.g., Monte Carlo simulation) to yield a probabilistic distribution of NPVs and 90% CIs for all strategies.

In [9]:
# Anthropic has a slightly different API, and Max Tokens is required

model_name = "claude-sonnet-4-5"

claude = Anthropic()
response = claude.messages.create(model=model_name, messages=messages, max_tokens=1000)
answer = response.content[0].text

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

TypeError: "Could not resolve authentication method. Expected either api_key or auth_token to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"

In [ ]:
gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
model_name = "gemini-2.5-flash"

response = gemini.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
deepseek = OpenAI(api_key=deepseek_api_key, base_url="https://api.deepseek.com/v1")
model_name = "deepseek-chat"

response = deepseek.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
# Updated with the latest Open Source model from OpenAI

groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
model_name = "openai/gpt-oss-120b"

response = groq.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)


## For the next cell, we will use Ollama

Ollama runs a local web service that gives an OpenAI compatible endpoint,  
and runs models locally using high performance C++ code.

If you don't have Ollama, install it here by visiting https://ollama.com then pressing Download and following the instructions.

After it's installed, you should be able to visit here: http://localhost:11434 and see the message "Ollama is running"

You might need to restart Cursor (and maybe reboot). Then open a Terminal (control+\`) and run `ollama serve`

Useful Ollama commands (run these in the terminal, or with an exclamation mark in this notebook):

`ollama pull <model_name>` downloads a model locally  
`ollama ls` lists all the models you've downloaded  
`ollama rm <model_name>` deletes the specified model from your downloads

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Super important - ignore me at your peril!</h2>
            <span style="color:#ff7800;">The model called <b>llama3.3</b> is FAR too large for home computers - it's not intended for personal computing and will consume all your resources! Stick with the nicely sized <b>llama3.2</b> or <b>llama3.2:1b</b> and if you want larger, try llama3.1 or smaller variants of Qwen, Gemma, Phi or DeepSeek. See the <A href="https://ollama.com/models">the Ollama models page</a> for a full list of models and sizes.
            </span>
        </td>
    </tr>
</table>

In [ ]:
!ollama pull llama3.2

In [ ]:
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
model_name = "llama3.2"

response = ollama.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
# So where are we?

print(competitors)
print(answers)


In [ ]:
# It's nice to know how to use "zip"
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")


In [ ]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

In [ ]:
print(together)

In [ ]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [ ]:
print(judge)

In [ ]:
judge_messages = [{"role": "user", "content": judge}]

In [ ]:
# Judgement time!

openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=judge_messages,
)
results = response.choices[0].message.content
print(results)


In [ ]:
# OK let's turn this into results!

results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Which pattern(s) did this use? Try updating this to add another Agentic design pattern.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">These kinds of patterns - to send a task to multiple models, and evaluate results,
            are common where you need to improve the quality of your LLM response. This approach can be universally applied
            to business projects where accuracy is critical.
            </span>
        </td>
    </tr>
</table>